# Amazon EKS에 Strands 여행 에이전트를 배포하고 AgentCore Observability 및 Evaluations 사용하기

이 노트북은 [Strands Agents SDK](https://github.com/strands-agents/sdk-python)로 구축한 여행 에이전트를 Amazon EKS에 배포하는 과정을 자동화합니다.

## 사전 요구 사항

- 설치 및 구성이 완료된 [AWS CLI](https://aws.amazon.com/cli/)
- 설치된 [eksctl](https://eksctl.io/installation/)(v0.208.x 이상)
- 설치된 [Helm](https://helm.sh/)(v3 이상)
- 설치된 [kubectl](https://docs.aws.amazon.com/eks/latest/userguide/install-kubectl.html)
- 설치되어 실행 중인 [Docker](https://www.docker.com/)
- AWS 계정에서 활성화된 Amazon Bedrock Claude 모델

In [ ]:
# macOS에서 사전 요구 사항을 설치하려면 주석을 해제하고 실행
# !brew tap weaveworks/tap
# !brew install weaveworks/tap/eksctl
# !brew install helm
# !brew install kubectl

# 설치 확인
!echo "=== Checking installed versions ==="
!aws --version
!eksctl version
!helm version --short
!kubectl version --client
!docker --version

## 1. 구성

배포에 사용할 환경 변수를 설정합니다. 필요에 따라 값을 수정하세요.

In [ ]:
import os

# AWS 계정 ID 자동 감지
account_id = !aws sts get-caller-identity --query 'Account' --output text
os.environ["AWS_ACCOUNT_ID"] = account_id[0]

# 사용자가 구성할 수 있는 설정(환경 변수에서 읽거나 기본값 사용)
# 배포를 사용자 지정하려면 실행 전에 이 환경 변수를 설정
os.environ["AWS_REGION"] = os.getenv("AWS_REGION", "us-east-1")
os.environ["CLUSTER_NAME"] = os.getenv("CLUSTER_NAME", "eks-strands-agents-demo")
os.environ["SERVICE_NAME"] = os.getenv("SERVICE_NAME", "strands-agents-travel")

# CloudWatch 구성
os.environ["LOG_GROUP_NAME"] = os.getenv("LOG_GROUP_NAME", "/strands-agents/travel")
os.environ["LOG_STREAM_NAME"] = os.getenv("LOG_STREAM_NAME", "agent-logs")
os.environ["METRIC_NAMESPACE"] = os.getenv("METRIC_NAMESPACE", "StrandsAgents/Travel")

# 포트 구성
os.environ["LOCAL_PORT"] = os.getenv("LOCAL_PORT", "8080")
os.environ["SERVICE_PORT"] = os.getenv("SERVICE_PORT", "80")

# 구성 표시
print("=== Deployment Configuration ===")
print(f"AWS Account ID: {os.environ['AWS_ACCOUNT_ID']}")
print(f"AWS Region: {os.environ['AWS_REGION']}")
print(f"Cluster Name: {os.environ['CLUSTER_NAME']}")
print(f"Service Name: {os.environ['SERVICE_NAME']}")
print(f"Log Group: {os.environ['LOG_GROUP_NAME']}")
print(f"Log Stream: {os.environ['LOG_STREAM_NAME']}")
print(f"Metric Namespace: {os.environ['METRIC_NAMESPACE']}")
print(f"Local Port: {os.environ['LOCAL_PORT']}")
print(f"Service Port: {os.environ['SERVICE_PORT']}")

## 2. CloudWatch 로그 그룹 생성

OpenTelemetry 로그를 위한 CloudWatch 로그 그룹과 로그 스트림을 생성합니다.

In [ ]:
%%bash
echo "Creating CloudWatch log group and stream..."
aws logs create-log-group --log-group-name ${LOG_GROUP_NAME} --region ${AWS_REGION} 2>/dev/null || echo "Log group already exists"
aws logs create-log-stream --log-group-name ${LOG_GROUP_NAME} --log-stream-name ${LOG_STREAM_NAME} --region ${AWS_REGION} 2>/dev/null || echo "Log stream already exists"
echo "CloudWatch resources ready!"

## 3. CloudWatch 구성으로 Dockerfile 업데이트

Dockerfile의 자리 표시자 값을 실제 CloudWatch 구성으로 바꿉니다.

In [ ]:
import os
import shutil

dockerfile_path = "docker/Dockerfile"
dockerfile_backup = "docker/Dockerfile.backup"

# 파일 존재 여부 확인
if not os.path.exists(dockerfile_path):
    raise FileNotFoundError(
        f"Dockerfile not found at {dockerfile_path}. Make sure you're running from the correct directory."
    )

# Dockerfile 읽기
with open(dockerfile_path, "r") as f:
    content = f.read()

# 자리 표시자 존재 여부 확인
if "<YOUR_LOG_GROUP>" in content or "<YOUR_SERVICE_NAME>" in content:
    # 수정 전에 백업 생성
    shutil.copy(dockerfile_path, dockerfile_backup)
    print(f"Backup created: {dockerfile_backup}")

    # 자리 표시자를 실제 값으로 변경
    content = content.replace("<YOUR_SERVICE_NAME>", os.environ["SERVICE_NAME"])
    content = content.replace("<YOUR_LOG_GROUP>", os.environ["LOG_GROUP_NAME"])
    content = content.replace("<YOUR_LOG_STREAM>", os.environ["LOG_STREAM_NAME"])
    content = content.replace("<YOUR_METRIC_NAMESPACE>", os.environ["METRIC_NAMESPACE"])

    # 변경 내용 쓰기
    with open(dockerfile_path, "w") as f:
        f.write(content)
    print("Dockerfile updated with configuration values")
else:
    print("Dockerfile already configured (no placeholders found)")
    print("To restore placeholders, copy Dockerfile.backup to Dockerfile")

# 업데이트된 OTEL 행 표시
print("\\nCurrent OTEL configuration:")
for line in content.split("\\n"):
    if "OTEL_RESOURCE_ATTRIBUTES" in line or "OTEL_EXPORTER_OTLP_LOGS_HEADERS" in line:
        print(f"  {line}")

## 4. EKS Cluster 생성

EKS Auto Mode cluster를 생성합니다. 이 단계는 약 15~20분이 걸립니다.

In [ ]:
%%bash
echo "Creating EKS Auto Mode cluster: $CLUSTER_NAME"
echo "This will take approximately 15-20 minutes..."

eksctl create cluster --name $CLUSTER_NAME --region $AWS_REGION --enable-auto-mode

In [ ]:
%%bash
# kubeconfig context 구성
aws eks update-kubeconfig --name $CLUSTER_NAME --region $AWS_REGION

# Cluster 접근 확인
echo "Cluster nodes:"
kubectl get nodes

## 5. Docker 이미지 빌드 및 ECR 푸시

여행 에이전트 Docker 이미지를 빌드하고 Amazon ECR로 푸시합니다.

`중요` - 로컬 Docker 인스턴스가 실행 중인지 확인하세요.

In [ ]:
%%bash
# Amazon ECR 인증
echo "Authenticating to ECR..."
aws ecr get-login-password --region ${AWS_REGION} | docker login --username AWS --password-stdin ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com

# ECR 리포지토리 생성(이미 있으면 오류 무시)
echo "Creating ECR repository..."
aws ecr create-repository --repository-name ${SERVICE_NAME} --region ${AWS_REGION} 2>/dev/null || echo "Repository already exists"

In [ ]:
%%bash
# Docker 이미지 빌드
echo "Building Docker image..."
docker build --platform linux/amd64 -t ${SERVICE_NAME}:latest docker/

# ECR용 이미지 태그 지정
docker tag ${SERVICE_NAME}:latest ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com/${SERVICE_NAME}:latest

# ECR로 이미지 푸시
echo "Pushing image to ECR..."
docker push ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com/${SERVICE_NAME}:latest

echo "Docker image pushed successfully!"

## 6. IAM 정책 구성

Amazon Bedrock 및 CloudWatch Logs 권한이 포함된 IAM 정책을 생성합니다.

In [ ]:
%%bash
# Bedrock 및 CloudWatch Logs 권한이 포함된 IAM 정책 생성
cat > /tmp/travel-agent-policy.json << 'EOF'
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:InvokeModelWithResponseStream"
      ],
      "Resource": "*"
    },
    {
      "Effect": "Allow",
      "Action": [
        "logs:CreateLogGroup",
        "logs:CreateLogStream",
        "logs:PutLogEvents",
        "logs:DescribeLogGroups",
        "logs:DescribeLogStreams"
      ],
      "Resource": "*"
    }
  ]
}
EOF

# IAM 정책 생성(이미 있으면 오류 무시)
aws iam create-policy \
  --policy-name ${SERVICE_NAME}-policy \
  --policy-document file:///tmp/travel-agent-policy.json 2>/dev/null || echo "Policy already exists"

rm -f /tmp/travel-agent-policy.json
echo "IAM policy ready!"

## 7. EKS Pod Identity 생성

서비스 계정에 대한 EKS Pod Identity 연결을 생성합니다.

In [ ]:
%%bash
# EKS Pod Identity 연결 생성
echo "Creating Pod Identity association..."
eksctl create podidentityassociation --cluster $CLUSTER_NAME \
  --namespace default \
  --service-account-name ${SERVICE_NAME} \
  --permission-policy-arns arn:aws:iam::${AWS_ACCOUNT_ID}:policy/${SERVICE_NAME}-policy \
  --role-name eks-${SERVICE_NAME} \
  --region $AWS_REGION

echo "Pod Identity association created!"

## 8. CloudWatch Observability Addon 설치(선택 사항)

> **참고:** 이 단계는 **선택 사항**입니다. Bedrock AgentCore Observability에는 CloudWatch Observability addon이 필요하지 않습니다. AgentCore는 Dockerfile의 OTEL 구성을 사용하여 텔레메트리를 CloudWatch로 직접 전송합니다. AgentCore 관측성만 필요한 경우 이 섹션을 건너뛰세요.

AgentCore 텔레메트리 외에 Kubernetes 수준의 추가 지표와 로그를 수집하려면 CloudWatch Observability addon을 설치합니다.

In [ ]:
%%bash
# CloudWatch agent용 Pod Identity 생성
echo "Creating CloudWatch agent Pod Identity..."
eksctl create podidentityassociation --cluster $CLUSTER_NAME \
  --namespace amazon-cloudwatch \
  --service-account-name cloudwatch-agent \
  --permission-policy-arns arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy \
  --role-name eks-cloudwatch-agent \
  --region $AWS_REGION

echo "CloudWatch agent Pod Identity created!"

In [ ]:
%%bash
# CloudWatch Observability addon 설치
echo "Installing CloudWatch Observability addon..."
aws eks create-addon \
  --addon-name amazon-cloudwatch-observability \
  --cluster-name $CLUSTER_NAME \
  --region $AWS_REGION

echo "Waiting for addon to be active..."
aws eks wait addon-active --cluster-name $CLUSTER_NAME --addon-name amazon-cloudwatch-observability --region $AWS_REGION
echo "CloudWatch Observability addon installed!"

## 9. Helm 차트 배포

Helm 차트를 사용하여 여행 에이전트 애플리케이션을 배포합니다.

In [ ]:
%%bash
# 차트 디렉터리 존재 여부 확인
if [ ! -d "./chart" ]; then
    echo "Error: Helm chart directory './chart' not found"
    echo "Make sure you're running from the strands-travel-agent-eks directory"
    exit 1
fi

# Helm으로 배포(멱등성을 위해 upgrade --install 사용)
echo "Deploying Travel Agent with Helm..."
helm upgrade --install ${SERVICE_NAME} ./chart \
  --set image.repository=${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com/${SERVICE_NAME} \
  --set image.tag=latest

echo "Helm deployment initiated!"

In [ ]:
%%bash
# 배포를 사용할 수 있을 때까지 대기
echo "Waiting for deployment to be ready..."
kubectl wait --for=condition=available deployments ${SERVICE_NAME} --timeout=300s

# Pod 상태 확인
echo "\nPod status:"
kubectl get pods -l app.kubernetes.io/name=${SERVICE_NAME}

## 10. 포트 포워딩 시작

로컬에서 여행 에이전트에 접근할 수 있도록 백그라운드에서 포트 포워딩을 시작합니다.

In [ ]:
import subprocess
import time
import os

local_port = os.environ.get("LOCAL_PORT", "8080")
service_port = os.environ.get("SERVICE_PORT", "80")
service_name = os.environ["SERVICE_NAME"]

# 백그라운드에서 포트 포워딩 시작
port_forward = subprocess.Popen(
    [
        "kubectl",
        "--namespace",
        "default",
        "port-forward",
        f"service/{service_name}",
        f"{local_port}:{service_port}",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(f"Port-forward started (PID: {port_forward.pid})")
print(f"Agent will be available at: http://localhost:{local_port}/travel")
print("\\nNote: Run the 'Stop Port Forward' cell below when done testing")
time.sleep(5)  # 포트 포워딩 연결 대기

## 11. 에이전트 테스트

테스트 쿼리로 여행 에이전트를 호출합니다.

In [ ]:
import requests
import os

local_port = os.environ.get("LOCAL_PORT", "8080")
url = f"http://localhost:{local_port}/travel"
payload = {"prompt": "What are the best places to visit in Tokyo in March?"}

print(f"Sending request to: {url}")
print(f"Prompt: {payload['prompt']}")
print("\\nWaiting for response (this may take a minute)...\\n")

try:
    response = requests.post(url, json=payload, timeout=120)
    print(f"Status: {response.status_code}")
    print(f"\\nResponse:\\n{response.text}")
except requests.exceptions.ConnectionError as e:
    print(f"Connection failed: {e}")
    print("\\nMake sure port-forward is running (run the cell above)")

## 12. 포트 포워딩 중지

테스트가 끝나면 포트 포워딩 프로세스를 중지합니다.

In [ ]:
# 포트 포워딩 프로세스 중지
if "port_forward" in dir() and port_forward.poll() is None:
    port_forward.terminate()
    print("Port-forward stopped")
else:
    print("Port-forward not running")

## 13. 리소스 정리(선택 사항)

이 노트북에서 생성한 모든 리소스를 제거하려면 아래 셀의 주석을 해제하고 실행하세요.

In [ ]:
# %%bash
# # Helm 차트 제거
# echo "Uninstalling helm chart..."
# helm uninstall ${SERVICE_NAME}

In [ ]:
# %%bash
# # EKS cluster 삭제(몇 분 정도 걸림)
# echo "Deleting EKS cluster: $CLUSTER_NAME"
# echo "This will take several minutes..."
# eksctl delete cluster --name $CLUSTER_NAME --region $AWS_REGION --wait

In [ ]:
# %%bash
# # IAM 정책 삭제
# echo "Deleting IAM policy..."
# aws iam delete-policy --policy-arn arn:aws:iam::${AWS_ACCOUNT_ID}:policy/${SERVICE_NAME}-policy

In [ ]:
# %%bash
# # ECR 리포지토리 삭제
# echo "Deleting ECR repository..."
# aws ecr delete-repository --repository-name ${SERVICE_NAME} --region ${AWS_REGION} --force

In [ ]:
# %%bash
# # CloudWatch 로그 그룹 삭제
# echo "Deleting CloudWatch log group..."
# aws logs delete-log-group --log-group-name ${LOG_GROUP_NAME} --region ${AWS_REGION}